In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import os

In [2]:
OLD_CSV = "/root/workspace/PlantCLEF2026/src_experiments/002_bioclip_tile_zero_shot_v2/data/PlantCLEF2024_single_plant_training_metadata.csv"
NEW_CSV = "/workspace/plantclef/processed/inat_research_grade_manifest_clean.csv"

OLD_IMG_ROOT = Path("/workspace/plantclef/raw/train/images_max_side_800")
OUT_DIR = Path("/workspace/plantclef/processed")

OUT_DIR.mkdir(parents=True, exist_ok=True)

MAX_IMAGES_PER_SPECIES = 1000
ALPHA = 0.5
SEED = 42

In [3]:
old_data = pd.read_csv(OLD_CSV, sep=";", low_memory=False)
new_data_clean = pd.read_csv(NEW_CSV, low_memory=False)

print("Old rows:", len(old_data))
print("New rows:", len(new_data_clean))

print("Old columns:")
print(old_data.columns.tolist())

print("\nNew columns:")
print(new_data_clean.columns.tolist())

Old rows: 1408033
New rows: 1245748
Old columns:
['image_name', 'organ', 'species_id', 'obs_id', 'license', 'partner', 'author', 'altitude', 'latitude', 'longitude', 'gbif_species_id', 'species', 'genus', 'family', 'dataset', 'publisher', 'references', 'url', 'learn_tag', 'image_backup_url']

New columns:
['image_path', 'species_id', 'gbif_species_id', 'gbif_occurrence_id', 'license', 'scientific_name', 'url']


In [4]:
old = old_data.copy()

old["species_id"] = pd.to_numeric(old["species_id"], errors="coerce").astype("Int64")
old["gbif_species_id"] = pd.to_numeric(old["gbif_species_id"], errors="coerce").astype("Int64")

old["image_path"] = old.apply(
    lambda row: str(
        OLD_IMG_ROOT
        / str(row["species_id"])
        / str(row["image_name"])
    ),
    axis=1,
)

old[["species_id", "image_name", "image_path"]].head()

,species_id,image_name,image_path
0,1396710,59feabe1c98f06e7f819f73c8246bd8f1a89556b.jpg,/workspace/plantclef/raw/train/images_max_side...
1,1396710,dc273995a89827437d447f29a52ccac86f65476e.jpg,/workspace/plantclef/raw/train/images_max_side...
2,1396710,416235e7023a4bd1513edf036b6097efc693a304.jpg,/workspace/plantclef/raw/train/images_max_side...
3,1396710,cbd18fade82c46a5c725f1f3d982174895158afc.jpg,/workspace/plantclef/raw/train/images_max_side...
4,1396710,f82c8c6d570287ebed8407cefcfcb2a51eaaf56e.jpg,/workspace/plantclef/raw/train/images_max_side...


In [5]:
old_std = pd.DataFrame({
    "image_path": old["image_path"],
    "image_name": old["image_name"],
    "species_id": old["species_id"],
    "gbif_species_id": old["gbif_species_id"],
    "scientific_name": old["species"],
    "license": old["license"],
    "source": "old_plantclef",
})

if "genus" in old.columns:
    old_std["genus"] = old["genus"]

if "family" in old.columns:
    old_std["family"] = old["family"]

if "url" in old.columns:
    old_std["url"] = old["url"]

old_std.head()

,image_path,image_name,species_id,gbif_species_id,scientific_name,license,source,genus,family,url
0,/workspace/plantclef/raw/train/images_max_side...,59feabe1c98f06e7f819f73c8246bd8f1a89556b.jpg,1396710,5284517,Taxus baccata L.,cc-by-sa,old_plantclef,Taxus,Taxaceae,https://bs.plantnet.org/image/o/59feabe1c98f06...
1,/workspace/plantclef/raw/train/images_max_side...,dc273995a89827437d447f29a52ccac86f65476e.jpg,1396710,5284517,Taxus baccata L.,cc-by-sa,old_plantclef,Taxus,Taxaceae,https://bs.plantnet.org/image/o/dc273995a89827...
2,/workspace/plantclef/raw/train/images_max_side...,416235e7023a4bd1513edf036b6097efc693a304.jpg,1396710,5284517,Taxus baccata L.,cc-by-sa,old_plantclef,Taxus,Taxaceae,https://bs.plantnet.org/image/o/416235e7023a4b...
3,/workspace/plantclef/raw/train/images_max_side...,cbd18fade82c46a5c725f1f3d982174895158afc.jpg,1396710,5284517,Taxus baccata L.,cc-by-sa,old_plantclef,Taxus,Taxaceae,https://bs.plantnet.org/image/o/cbd18fade82c46...
4,/workspace/plantclef/raw/train/images_max_side...,f82c8c6d570287ebed8407cefcfcb2a51eaaf56e.jpg,1396710,5284517,Taxus baccata L.,cc-by-sa,old_plantclef,Taxus,Taxaceae,https://bs.plantnet.org/image/o/f82c8c6d570287...


In [6]:
new = new_data_clean.copy()

new["species_id"] = pd.to_numeric(new["species_id"], errors="coerce").astype("Int64")
new["gbif_species_id"] = pd.to_numeric(new["gbif_species_id"], errors="coerce").astype("Int64")

new_std = pd.DataFrame({
    "image_path": new["image_path"],
    "image_name": new["image_path"].astype(str).apply(lambda x: Path(x).name),
    "species_id": new["species_id"],
    "gbif_species_id": new["gbif_species_id"],
    "scientific_name": new["scientific_name"],
    "license": new["license"],
    "source": "new_inat",
})

if "url" in new.columns:
    new_std["url"] = new["url"]

if "gbif_occurrence_id" in new.columns:
    new_std["gbif_occurrence_id"] = new["gbif_occurrence_id"]

new_std.head()

,image_path,image_name,species_id,gbif_species_id,scientific_name,license,source,url,gbif_occurrence_id
0,/workspace/plantclef/raw/inat_research_grade/1...,5938299218_605069874.jpg,1396710,5284517,Taxus baccata L.,http://creativecommons.org/licenses/by-nc/4.0/...,new_inat,https://inaturalist-open-data.s3.amazonaws.com...,5938299218
1,/workspace/plantclef/raw/inat_research_grade/1...,5938200063_605359819.jpg,1396710,5284517,Taxus baccata L.,http://creativecommons.org/licenses/by-nc/4.0/...,new_inat,https://inaturalist-open-data.s3.amazonaws.com...,5938200063
2,/workspace/plantclef/raw/inat_research_grade/1...,5938168811_604750178.jpg,1396710,5284517,Taxus baccata L.,http://creativecommons.org/publicdomain/zero/1...,new_inat,https://inaturalist-open-data.s3.amazonaws.com...,5938168811
3,/workspace/plantclef/raw/inat_research_grade/1...,5938071098_605001704.jpg,1396710,5284517,Taxus baccata L.,http://creativecommons.org/licenses/by/4.0/leg...,new_inat,https://inaturalist-open-data.s3.amazonaws.com...,5938071098
4,/workspace/plantclef/raw/inat_research_grade/1...,5938251864_604497369.jpg,1396710,5284517,Taxus baccata L.,http://creativecommons.org/licenses/by/4.0/leg...,new_inat,https://inaturalist-open-data.s3.amazonaws.com...,5938251864


In [7]:
combined = pd.concat([old_std, new_std], ignore_index=True)

combined = combined.dropna(subset=["species_id", "image_path"]).copy()

combined["species_id"] = pd.to_numeric(
    combined["species_id"], errors="coerce"
).astype("Int64")

combined["gbif_species_id"] = pd.to_numeric(
    combined["gbif_species_id"], errors="coerce"
).astype("Int64")

before = len(combined)
combined = combined.drop_duplicates(subset=["image_path"]).copy()
after = len(combined)

print("Combined rows:", len(combined))
print("Combined species:", combined["species_id"].nunique())
print("Removed duplicate image_path rows:", before - after)

combined.head()

Combined rows: 2653781
Combined species: 7806
Removed duplicate image_path rows: 0


,image_path,image_name,species_id,gbif_species_id,scientific_name,license,source,genus,family,url,gbif_occurrence_id
0,/workspace/plantclef/raw/train/images_max_side...,59feabe1c98f06e7f819f73c8246bd8f1a89556b.jpg,1396710,5284517,Taxus baccata L.,cc-by-sa,old_plantclef,Taxus,Taxaceae,https://bs.plantnet.org/image/o/59feabe1c98f06...,NaN
1,/workspace/plantclef/raw/train/images_max_side...,dc273995a89827437d447f29a52ccac86f65476e.jpg,1396710,5284517,Taxus baccata L.,cc-by-sa,old_plantclef,Taxus,Taxaceae,https://bs.plantnet.org/image/o/dc273995a89827...,NaN
2,/workspace/plantclef/raw/train/images_max_side...,416235e7023a4bd1513edf036b6097efc693a304.jpg,1396710,5284517,Taxus baccata L.,cc-by-sa,old_plantclef,Taxus,Taxaceae,https://bs.plantnet.org/image/o/416235e7023a4b...,NaN
3,/workspace/plantclef/raw/train/images_max_side...,cbd18fade82c46a5c725f1f3d982174895158afc.jpg,1396710,5284517,Taxus baccata L.,cc-by-sa,old_plantclef,Taxus,Taxaceae,https://bs.plantnet.org/image/o/cbd18fade82c46...,NaN
4,/workspace/plantclef/raw/train/images_max_side...,f82c8c6d570287ebed8407cefcfcb2a51eaaf56e.jpg,1396710,5284517,Taxus baccata L.,cc-by-sa,old_plantclef,Taxus,Taxaceae,https://bs.plantnet.org/image/o/f82c8c6d570287...,NaN


In [8]:
full_out = OUT_DIR / "combined_old_new_manifest_full.csv"

combined.to_csv(full_out, index=False)

print("Saved:", full_out)
print("Rows:", len(combined))

Saved: /workspace/plantclef/processed/combined_old_new_manifest_full.csv
Rows: 2653781


In [9]:
combined_counts = (
    combined.groupby("species_id")
    .agg(
        n_images=("image_path", "count"),
        n_old=("source", lambda x: int((x == "old_plantclef").sum())),
        n_new=("source", lambda x: int((x == "new_inat").sum())),
        scientific_name=("scientific_name", "first"),
        gbif_species_id=("gbif_species_id", "first"),
    )
    .reset_index()
    .sort_values("n_images", ascending=True)
)

combined_counts.head(50)

,species_id,n_images,n_old,n_new,scientific_name,gbif_species_id
7756,1744551,1,1,0,"Helianthemum dianicum Pérez Dacosta, M.B.Cresp...",8418210
7743,1744511,1,1,0,Genista tribracteolata (Webb) Pau,5354550
5806,1457840,1,1,0,Ferulago brachyloba Boiss. & Reut.,3635847
6540,1647166,1,1,0,Armeria muelleri A.Huet,5668284
7805,1744937,1,1,0,"Linaria semialata D.López, Sánchez-Gómez, J.F....",11164302
7731,1744480,1,1,0,Armeria cantabrica Boiss. & Reut. ex Willk.,7298203
3782,1392852,1,1,0,Hieracium chaixianum Arv.-Touv. & Gaut.,4229027
6532,1647141,1,1,0,Armeria castellana Boiss. & Reut. ex Leresche,7683925
3959,1393553,1,1,0,Limonium corsicum Erben,4088422
6503,1646226,1,1,0,Linaria bubanii Font Quer,8423889


In [10]:
combined_counts["n_images"].describe(
    percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
)

count    7806.000000
mean      339.966820
std       308.217273
min         1.000000
1%          1.000000
5%          4.000000
10%         7.000000
25%        41.250000
50%       249.000000
75%       603.000000
90%       791.000000
95%       883.000000
99%      1012.000000
max      1323.000000
Name: n_images, dtype: float64

In [15]:
bins = [0, 1, 2, 5, 10, 20, 50, 100, 250, 500, 1000, 2000, 10_000_000]
labels = [
    "1",
    "2",
    "3-5",
    "6-10",
    "11-20",
    "21-50",
    "51-100",
    "101-250",
    "251-500",
    "501-1000",
    "1001-2000",
    ">2000",
]

combined_counts["count_bin"] = pd.cut(
    combined_counts["n_images"],
    bins=bins,
    labels=labels,
    include_lowest=True,
)

combined_counts["count_bin"].value_counts().sort_index()

count_bin
1             145
2             109
3-5           344
6-10          381
11-20         437
21-50         701
51-100        584
101-250      1217
251-500      1175
501-1000     2596
1001-2000     117
>2000           0
Name: count, dtype: int64

In [13]:
def cap_per_species(df, max_images_per_species=1000, seed=42):
    if max_images_per_species <= 0:
        return df.copy()

    return (
        df.groupby("species_id", group_keys=False)
        .apply(
            lambda x: x.sample(
                min(len(x), max_images_per_species),
                random_state=seed,
            )
        )
        .reset_index(drop=True)
    )

In [17]:
combined_capped = cap_per_species(
    combined,
    max_images_per_species=MAX_IMAGES_PER_SPECIES,
    seed=SEED,
)

print("Before cap:", len(combined))
print("After cap:", len(combined_capped))
print("Species:", combined_capped["gbif_species_id"].nunique())

combined_capped.head()

Before cap: 2653781
After cap: 2648507
Species: 7796


,image_path,image_name,gbif_species_id,scientific_name,license,source,genus,family,url,gbif_occurrence_id
0,/workspace/plantclef/raw/inat_research_grade/1...,6131142156_488316785.jpg,3140346,Lactuca virosa L.,http://creativecommons.org/licenses/by-nc/4.0/...,new_inat,NaN,NaN,https://inaturalist-open-data.s3.amazonaws.com...,6.131142e+09
1,/workspace/plantclef/raw/train/images_max_side...,9eef7b311942028df2360b46304163e2982b2c79.jpg,3140346,Lactuca virosa L.,cc-by-sa,old_plantclef,Lactuca,Asteraceae,https://bs.plantnet.org/image/o/9eef7b31194202...,NaN
2,/workspace/plantclef/raw/train/images_max_side...,069e0613293fd6e69a7aa20b85a5277bec501cef.jpg,3140346,Lactuca virosa L.,cc-by-sa,old_plantclef,Lactuca,Asteraceae,https://bs.plantnet.org/image/o/069e0613293fd6...,NaN
3,/workspace/plantclef/raw/train/images_max_side...,ef462e6ce509a5e8d86667d38a7f4bcea120e500.jpg,3140346,Lactuca virosa L.,cc-by-sa,old_plantclef,Lactuca,Asteraceae,https://bs.plantnet.org/image/o/ef462e6ce509a5...,NaN
4,/workspace/plantclef/raw/train/images_max_side...,d535eb271c4008a9d19f80f68b87c631573c0161.jpg,3140346,Lactuca virosa L.,cc-by-sa,old_plantclef,Lactuca,Asteraceae,https://bs.plantnet.org/image/o/d535eb271c4008...,NaN


In [20]:
capped_counts = (
    combined_capped.groupby("gbif_species_id")
    .agg(
        n_images=("image_path", "count"),
        n_old=("source", lambda x: int((x == "old_plantclef").sum())),
        n_new=("source", lambda x: int((x == "new_inat").sum())),
        scientific_name=("scientific_name", "first"),
    )
    .reset_index()
    .sort_values("n_images", ascending=True)
)

capped_counts.head(50)

,gbif_species_id,n_images,n_old,n_new,scientific_name
7748,11072581,1,1,0,Aster bellidiastrum (L.) Scop.
7784,12234790,1,1,0,Verbascum × kerneri Borbás
7782,12228878,1,1,0,Cynanchica × jordanii (E.P.Perrier & Songeon) ...
7762,11418923,1,1,0,Aconitum vivantii Rottenst.
4234,4258625,1,1,0,Asplenium majoricum Litard.
7754,11164302,1,1,0,"Linaria semialata D.López, Sánchez-Gómez, J.F...."
6381,7313107,1,1,0,Isoetes boryana Durieu
6441,7331429,1,1,0,Helianthemum canum (L.) Hornem.
6562,7415504,1,1,0,Salicornia disarticulata Moss
6776,7683925,1,1,0,Armeria castellana Boiss. & Reut. ex Leresche
